In [1]:
!pip install konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.6/488.6 kB 24.8 MB/s eta 0:00:00


In [2]:
import json
from google.colab import drive
from konlpy.tag import Okt
from transformers import MarianMTModel, MarianTokenizer

In [3]:
# Google Drive와 연결
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Hugging Face 모델 및 토크나이저 로드
model_name = "Helsinki-NLP/opus-mt-ko-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to("cuda")  # GPU 사용

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/842k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/813k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.72M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [5]:
# 형태소 분석기 초기화
okt = Okt()

# 자립명사를 추출하는 함수
def extract_independent_nouns(sentence):
    tokens = okt.pos(sentence)
    nouns = [word for word, pos in tokens if pos == "Noun"]
    return nouns

# Hugging Face 모델을 사용하여 영어로 번역
def translate_with_opus_mt(nouns):
    translations = {}
    for noun in nouns:
        try:
            # 번역 입력 준비
            inputs = tokenizer(noun, return_tensors="pt", padding=True, truncation=True).to("cuda")
            # 번역 수행
            outputs = model.generate(**inputs)
            translated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
            translations[noun] = translated_text
        except Exception:
            translations[noun] = noun  # 번역 실패 시 원문 유지
    return translations

# 코드스위칭 문장을 생성하는 함수
def code_switching(sentence, translations):
    for noun, translation in translations.items():
        sentence = sentence.replace(noun, translation)
    return sentence

In [6]:
# 파일 경로 설정
input_file_path = '/content/drive/My Drive/파인마이닝 project/raw_conversation.jsonl'  # JSONL 파일 경로
output_file_path = '/content/drive/My Drive/파인마이닝 project/code_switched_dataset.jsonl'  # 결과 저장 경로

In [7]:
# output 부분만 처리
data = []
with open(input_file_path, 'r', encoding='utf-8') as file:
    for line in file:
        item = json.loads(line)
        if 'output' in item:
            data.append(item['output'])

In [10]:
len(data)

2063

In [11]:
# 코드스위칭된 결과 생성
code_switched_results = []
for sentence in data:
    # 자립명사 추출
    nouns = extract_independent_nouns(sentence)
    # 번역
    translations = translate_with_opus_mt(nouns)
    # 코드스위칭
    code_switched_sentence = code_switching(sentence, translations)
    # 결과 저장
    code_switched_results.append({"original_sentence": sentence, "code_switched_sentence": code_switched_sentence})

In [12]:
# 결과 저장
with open(output_file_path, 'w', encoding='utf-8') as file:
    for item in code_switched_results:
        json.dump(item, file, ensure_ascii=False)
        file.write('\n')

print(f"code-switched outputs saved to {output_file_path}")

code-switched outputs saved to /content/drive/My Drive/파인마이닝 project/code_switched_dataset.jsonl


In [14]:
with open("/content/drive/My Drive/파인마이닝 project/code_switched_dataset.jsonl", 'r', encoding='utf-8') as file:
    data = [json.loads(line) for line in file]

print(data[30])

{'original_sentence': '죄송합니다. 저는 강아지를 키우지 않으므로 해당 질문에 대한 답변을 제공할 수 없습니다.', 'code_switched_sentence': '죄송합니다. Hey.는 Puppy를 키우지 않으므로 Equivalent Question에 About Answer을 Presentation할 Number 없습니다.'}
